# 📈 Production Quantity Analysis - Deep Dive

This notebook provides detailed comparison of production quantities by:
- **Crop Type**
- **Irrigation System**
- **Combined Analysis** (Crop × Irrigation System)

---

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure settings
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded successfully!")

In [ ]:
# Load dataset
df = pd.read_excel('Irrigation_DS.xlsx')

print("✓ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

## 2. Data Preparation

In [ ]:
# Verify required columns exist
required_cols = ['Production_Quantity_Baskets_Kg', 'Crop_Type', 'Irrigation_System_Type']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"⚠ Warning: Missing columns: {missing_cols}")
else:
    print("✓ All required columns present!")

# Display sample data
print("\nSample data:")
print("=" * 100)
display(df[['Crop_Type', 'Irrigation_System_Type', 'Production_Quantity_Baskets_Kg']].head(10))

In [ ]:
# Check data quality for production quantity
print("Production Quantity - Data Quality Check:")
print("=" * 100)

prod_col = 'Production_Quantity_Baskets_Kg'

if prod_col in df.columns:
    print(f"\nNull values: {df[prod_col].isnull().sum()}")
    print(f"Zero values: {(df[prod_col] == 0).sum()}")
    print(f"Negative values: {(df[prod_col] < 0).sum()}")
    print(f"\nMin value: {df[prod_col].min()}")
    print(f"Max value: {df[prod_col].max()}")
    print(f"Mean value: {df[prod_col].mean():.2f}")
    print(f"Median value: {df[prod_col].median():.2f}")

## 3. Production Quantity by Crop Type

In [ ]:
# Statistical summary by crop type
print("PRODUCTION QUANTITY BY CROP TYPE")
print("=" * 100)

if 'Crop_Type' in df.columns and 'Production_Quantity_Baskets_Kg' in df.columns:
    prod_by_crop = df.groupby('Crop_Type')['Production_Quantity_Baskets_Kg'].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max'),
        ('Total', 'sum')
    ]).round(2)
    
    display(prod_by_crop)
    
    # Save for later use
    prod_by_crop_stats = prod_by_crop

In [ ]:
# Visualization 1: Average Production by Crop Type
plt.figure(figsize=(12, 6))

crop_avg = df.groupby('Crop_Type')['Production_Quantity_Baskets_Kg'].mean().sort_values(ascending=False)

colors = plt.cm.viridis(np.linspace(0, 1, len(crop_avg)))
bars = plt.bar(range(len(crop_avg)), crop_avg.values, color=colors, edgecolor='black', linewidth=1.5)

plt.xticks(range(len(crop_avg)), crop_avg.index, rotation=45, ha='right')
plt.xlabel('Crop Type', fontsize=12, fontweight='bold')
plt.ylabel('Average Production (Baskets/Kg)', fontsize=12, fontweight='bold')
plt.title('Average Production Quantity by Crop Type', fontsize=14, fontweight='bold', pad=20)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, crop_avg.values)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(crop_avg)*0.02, 
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n🏆 Highest production: {crop_avg.index[0]} ({crop_avg.values[0]:.2f} baskets/kg)")
print(f"📊 Lowest production: {crop_avg.index[-1]} ({crop_avg.values[-1]:.2f} baskets/kg)")

In [ ]:
# Visualization 2: Box Plot - Production Distribution by Crop Type
plt.figure(figsize=(14, 7))

df.boxplot(column='Production_Quantity_Baskets_Kg', by='Crop_Type', 
           patch_artist=True, figsize=(14, 7))

plt.suptitle('')  # Remove default title
plt.title('Production Quantity Distribution by Crop Type', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Crop Type', fontsize=12, fontweight='bold')
plt.ylabel('Production Quantity (Baskets/Kg)', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Total Production by Crop Type (Pie Chart)
plt.figure(figsize=(10, 10))

total_prod_by_crop = df.groupby('Crop_Type')['Production_Quantity_Baskets_Kg'].sum().sort_values(ascending=False)

colors = plt.cm.Set3(range(len(total_prod_by_crop)))
plt.pie(total_prod_by_crop.values, labels=total_prod_by_crop.index, autopct='%1.1f%%',
        startangle=90, colors=colors, textprops={'fontsize': 11, 'fontweight': 'bold'})

plt.title('Total Production Share by Crop Type', fontsize=14, fontweight='bold', pad=20)
plt.axis('equal')
plt.tight_layout()
plt.show()

print("\nTotal Production by Crop:")
for crop, total in total_prod_by_crop.items():
    pct = (total / total_prod_by_crop.sum()) * 100
    print(f"  • {crop}: {total:.2f} baskets/kg ({pct:.1f}%)")

## 4. Production Quantity by Irrigation System

In [ ]:
# Statistical summary by irrigation system
print("PRODUCTION QUANTITY BY IRRIGATION SYSTEM")
print("=" * 100)

if 'Irrigation_System_Type' in df.columns and 'Production_Quantity_Baskets_Kg' in df.columns:
    prod_by_irrigation = df.groupby('Irrigation_System_Type')['Production_Quantity_Baskets_Kg'].agg([
        ('Count', 'count'),
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std', 'std'),
        ('Min', 'min'),
        ('Max', 'max'),
        ('Total', 'sum')
    ]).round(2)
    
    display(prod_by_irrigation)
    
    # Save for later use
    prod_by_irrigation_stats = prod_by_irrigation

In [ ]:
# Visualization 4: Comparison of Production by Irrigation System
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart - Average
irrigation_avg = df.groupby('Irrigation_System_Type')['Production_Quantity_Baskets_Kg'].mean()
colors_irr = ['#2E86AB', '#A23B72']

bars1 = axes[0].bar(range(len(irrigation_avg)), irrigation_avg.values, 
                    color=colors_irr, edgecolor='black', linewidth=1.5)
axes[0].set_xticks(range(len(irrigation_avg)))
axes[0].set_xticklabels(irrigation_avg.index, fontsize=11, fontweight='bold')
axes[0].set_ylabel('Average Production (Baskets/Kg)', fontsize=12, fontweight='bold')
axes[0].set_title('Average Production by Irrigation System', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Box plot
irrigation_data = [df[df['Irrigation_System_Type'] == system]['Production_Quantity_Baskets_Kg'].values 
                   for system in irrigation_avg.index]

bp = axes[1].boxplot(irrigation_data, labels=irrigation_avg.index, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_irr):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('Production Quantity (Baskets/Kg)', fontsize=12, fontweight='bold')
axes[1].set_title('Production Distribution by Irrigation System', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Performance comparison
if len(irrigation_avg) == 2:
    diff = irrigation_avg.values[0] - irrigation_avg.values[1]
    pct_diff = (diff / irrigation_avg.values[1]) * 100
    higher = irrigation_avg.index[0] if diff > 0 else irrigation_avg.index[1]
    print(f"\n📊 {higher} produces {abs(pct_diff):.1f}% more on average")

In [ ]:
# Visualization 5: Violin Plot - Production by Irrigation System
plt.figure(figsize=(12, 7))

sns.violinplot(data=df, x='Irrigation_System_Type', y='Production_Quantity_Baskets_Kg',
               palette='Set2', inner='box')

plt.xlabel('Irrigation System Type', fontsize=12, fontweight='bold')
plt.ylabel('Production Quantity (Baskets/Kg)', fontsize=12, fontweight='bold')
plt.title('Production Quantity Distribution by Irrigation System (Violin Plot)', 
          fontsize=14, fontweight='bold', pad=20)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Combined Analysis: Crop Type × Irrigation System

In [ ]:
# Pivot table: Production by Crop and Irrigation System
print("PRODUCTION QUANTITY: CROP TYPE × IRRIGATION SYSTEM")
print("=" * 100)

pivot_table = df.pivot_table(
    values='Production_Quantity_Baskets_Kg',
    index='Crop_Type',
    columns='Irrigation_System_Type',
    aggfunc=['mean', 'count', 'sum'],
    margins=True
).round(2)

print("\nAverage Production (Mean):")
display(pivot_table['mean'])

print("\nSample Count:")
display(pivot_table['count'])

print("\nTotal Production (Sum):")
display(pivot_table['sum'])

In [ ]:
# Visualization 6: Grouped Bar Chart - Crop × Irrigation
pivot_mean = df.pivot_table(
    values='Production_Quantity_Baskets_Kg',
    index='Crop_Type',
    columns='Irrigation_System_Type',
    aggfunc='mean'
)

plt.figure(figsize=(14, 7))
pivot_mean.plot(kind='bar', color=['#2E86AB', '#A23B72'], 
                edgecolor='black', linewidth=1.2, width=0.8)

plt.xlabel('Crop Type', fontsize=12, fontweight='bold')
plt.ylabel('Average Production (Baskets/Kg)', fontsize=12, fontweight='bold')
plt.title('Average Production: Crop Type × Irrigation System', fontsize=14, fontweight='bold', pad=20)
plt.legend(title='Irrigation System', title_fontsize=11, fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 7: Heatmap - Production by Crop and Irrigation
plt.figure(figsize=(10, 8))

sns.heatmap(pivot_mean, annot=True, fmt='.2f', cmap='YlGnBu', 
            cbar_kws={'label': 'Average Production (Baskets/Kg)'},
            linewidths=2, linecolor='white')

plt.xlabel('Irrigation System Type', fontsize=12, fontweight='bold')
plt.ylabel('Crop Type', fontsize=12, fontweight='bold')
plt.title('Production Heatmap: Crop Type × Irrigation System', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 8: Side-by-side comparison for each crop
crops = df['Crop_Type'].unique()
n_crops = len(crops)

fig, axes = plt.subplots(1, n_crops, figsize=(5*n_crops, 6), sharey=True)

if n_crops == 1:
    axes = [axes]

for idx, crop in enumerate(crops):
    crop_data = df[df['Crop_Type'] == crop]
    
    prod_by_sys = crop_data.groupby('Irrigation_System_Type')['Production_Quantity_Baskets_Kg'].mean()
    
    colors_crop = ['#2E86AB', '#A23B72']
    bars = axes[idx].bar(range(len(prod_by_sys)), prod_by_sys.values, 
                        color=colors_crop[:len(prod_by_sys)], edgecolor='black', linewidth=1.5)
    
    axes[idx].set_xticks(range(len(prod_by_sys)))
    axes[idx].set_xticklabels(prod_by_sys.index, rotation=45, ha='right')
    axes[idx].set_title(f'{crop}', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.1f}', ha='center', va='bottom', fontweight='bold')
    
    if idx == 0:
        axes[idx].set_ylabel('Average Production (Baskets/Kg)', fontsize=11, fontweight='bold')

fig.suptitle('Production Comparison by Irrigation System for Each Crop', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Statistical Tests

In [ ]:
# T-test or ANOVA to check if differences are statistically significant
from scipy import stats

print("STATISTICAL SIGNIFICANCE TESTS")
print("=" * 100)

# Test for irrigation systems
if 'Irrigation_System_Type' in df.columns:
    irrigation_types = df['Irrigation_System_Type'].unique()
    
    if len(irrigation_types) == 2:
        # T-test for two groups
        group1 = df[df['Irrigation_System_Type'] == irrigation_types[0]]['Production_Quantity_Baskets_Kg'].dropna()
        group2 = df[df['Irrigation_System_Type'] == irrigation_types[1]]['Production_Quantity_Baskets_Kg'].dropna()
        
        t_stat, p_value = stats.ttest_ind(group1, group2)
        
        print(f"\nT-test: Production by Irrigation System")
        print(f"  • {irrigation_types[0]} vs {irrigation_types[1]}")
        print(f"  • t-statistic: {t_stat:.4f}")
        print(f"  • p-value: {p_value:.4f}")
        
        if p_value < 0.05:
            print(f"  • ✓ Statistically significant difference (p < 0.05)")
        else:
            print(f"  • No statistically significant difference (p >= 0.05)")

# ANOVA for crop types
if 'Crop_Type' in df.columns:
    crop_groups = [df[df['Crop_Type'] == crop]['Production_Quantity_Baskets_Kg'].dropna() 
                   for crop in df['Crop_Type'].unique()]
    
    if len(crop_groups) > 2:
        f_stat, p_value = stats.f_oneway(*crop_groups)
        
        print(f"\nANOVA: Production by Crop Type")
        print(f"  • F-statistic: {f_stat:.4f}")
        print(f"  • p-value: {p_value:.4f}")
        
        if p_value < 0.05:
            print(f"  • ✓ Statistically significant difference between crops (p < 0.05)")
        else:
            print(f"  • No statistically significant difference between crops (p >= 0.05)")

## 7. Key Findings Summary

In [ ]:
print("=" * 100)
print("KEY FINDINGS: PRODUCTION QUANTITY ANALYSIS")
print("=" * 100)

# By Crop Type
print("\n📊 BY CROP TYPE:")
print("-" * 100)

crop_avg = df.groupby('Crop_Type')['Production_Quantity_Baskets_Kg'].mean().sort_values(ascending=False)
print(f"\n🏆 Highest producing crop: {crop_avg.index[0]}")
print(f"   Average production: {crop_avg.values[0]:.2f} baskets/kg")

print(f"\n📉 Lowest producing crop: {crop_avg.index[-1]}")
print(f"   Average production: {crop_avg.values[-1]:.2f} baskets/kg")

print(f"\nProduction range: {crop_avg.values[-1]:.2f} - {crop_avg.values[0]:.2f} baskets/kg")
print(f"Difference: {(crop_avg.values[0] - crop_avg.values[-1]):.2f} baskets/kg ({((crop_avg.values[0] / crop_avg.values[-1] - 1) * 100):.1f}% higher)")

# By Irrigation System
print("\n💧 BY IRRIGATION SYSTEM:")
print("-" * 100)

irr_avg = df.groupby('Irrigation_System_Type')['Production_Quantity_Baskets_Kg'].mean().sort_values(ascending=False)

for i, (system, avg_prod) in enumerate(irr_avg.items(), 1):
    count = df[df['Irrigation_System_Type'] == system].shape[0]
    print(f"\n{i}. {system}:")
    print(f"   Average production: {avg_prod:.2f} baskets/kg")
    print(f"   Number of farms: {count}")

if len(irr_avg) == 2:
    diff = irr_avg.values[0] - irr_avg.values[1]
    pct_diff = (diff / irr_avg.values[1]) * 100
    print(f"\nPerformance difference: {abs(pct_diff):.1f}%")
    print(f"{irr_avg.index[0]} produces {abs(diff):.2f} more baskets/kg on average")

# Best combination
print("\n🌟 BEST COMBINATION:")
print("-" * 100)

combo_avg = df.groupby(['Crop_Type', 'Irrigation_System_Type'])['Production_Quantity_Baskets_Kg'].mean().sort_values(ascending=False)

if len(combo_avg) > 0:
    best_combo = combo_avg.index[0]
    best_prod = combo_avg.values[0]
    print(f"\nCrop: {best_combo[0]}")
    print(f"Irrigation: {best_combo[1]}")
    print(f"Average production: {best_prod:.2f} baskets/kg")

print("\n" + "=" * 100)

## 8. Recommendations

### 📌 Based on the Production Quantity Analysis:

#### 🎯 Crop Selection
- Focus on **high-producing crops** identified in the analysis
- Consider market demand alongside production capacity
- Diversify with a mix of high and medium producers

#### 💧 Irrigation System Optimization
- If one system shows significantly higher production:
  - Evaluate cost-benefit of adopting that system
  - Consider water availability and infrastructure costs
  - Provide training/support for farmers to switch systems

#### 🔄 Crop-Irrigation Matching
- Use the **best combinations** identified in the analysis
- Match specific crops with their optimal irrigation system
- Create crop-specific irrigation guidelines

#### 📊 Further Analysis Needed
- Cost-per-kg analysis (production quantity vs total costs)
- Profitability analysis (production × market price − costs)
- Water efficiency (production per liter of water used)
- Time-based trends if multi-season data available

---

**End of Production Quantity Analysis** 🎉